In [1]:
import pandas as pd

In [2]:
#Define file paths

file_hicp = 'prc_hicp_manr.csv'   # Inflation
file_unemp = 'ei_lmhr_m.csv'      # Unemployment
file_cci = 'ei_bsco_m.csv'        # Consumer Confidence

In [3]:
#Create a helper function to clean and extract just what we need from each Eurostat file

def clean_eurostat_csv(filepath, new_value_col_name):
    # Load the CSV
    df = pd.read_csv(filepath)
    
    # Eurostat sometimes uses 'TIME_PERIOD' and 'OBS_VALUE' as standard column names
    # If your CSV uses lowercase (time_period, obs_value), pandas will adjust if we clean column names
    df.columns = df.columns.str.strip().str.upper()
    
    # Extract only the date and the actual value
    df_clean = df[['TIME_PERIOD', 'OBS_VALUE']].copy()
    
    # Rename columns to match the format needed for the H&M data merge
    df_clean = df_clean.rename(columns={
        'TIME_PERIOD': 'year_month',
        'OBS_VALUE': new_value_col_name
    })
    
    # Ensure year_month is treated as a string to avoid merging issues later
    df_clean['year_month'] = df_clean['year_month'].astype(str)
    
    return df_clean

In [4]:
#Apply the cleaning function to all three datasets

df_hicp_clean = clean_eurostat_csv(file_hicp, 'eurozone_hicp')
df_unemp_clean = clean_eurostat_csv(file_unemp, 'eurozone_unemployment_rate')
df_cci_clean = clean_eurostat_csv(file_cci, 'eurozone_cci')

In [5]:
#Merge them together sequentially
# Merge HICP and Unemployment

df_macro = pd.merge(df_hicp_clean, df_unemp_clean, on='year_month', how='inner')

In [6]:
# Merge the result with Consumer Confidence
df_macro = pd.merge(df_macro, df_cci_clean, on='year_month', how='inner')

In [7]:
#Check for missing values or anomalies

print(df_macro.head())
print(f"Total months matched: {len(df_macro)}")

  year_month  eurozone_hicp  eurozone_unemployment_rate  eurozone_cci
0    2018-09            2.1                         7.3          -4.7
1    2018-09            2.1                         7.3          -4.8
2    2018-09            2.1                         7.3          -2.7
3    2018-09            2.1                         7.3          -3.2
4    2018-09            2.1                         7.3           1.5
Total months matched: 27000


In [8]:
#Save the final clean dataset!

df_macro.to_csv('eurostat_macro_clean.csv', index=False)
print("Merge complete. Saved as 'eurostat_macro_clean.csv'")

Merge complete. Saved as 'eurostat_macro_clean.csv'
